# Konu 9 — Video üretimi

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 8'de görselleri koddan ve toplu olarak ürettin. Bugün görsele **zaman** ekliyoruz: kısa video klipler.
Videoyu **arayüzde** üreteceksin (Google Flow; adım adım anlatımı `ders_notu.md`'de). Görselde kullandığımız Cloudflare'de
video modeli yok; Google'ın video API'si ise ücretli: koddan üretmek her saniye için para ödemek demek. Bu defterin iki işi var:

1. **Maliyet.** Aynı klip ücretli bir API'yle koddan üretilseydi kaça mal olurdu? Süre ve çözünürlük fiyatı nasıl değiştiriyor?
2. **Kayıt.** Arayüzde ürettiğin her klibin yönergesini ve ayarlarını bir JSON günlüğüne yazmak (Konu 8'deki üretim günlüğü gibi).

Bu defter **anahtar istemez, internete çıkmaz.** Arayüze kendi kişisel Google hesabınla gireceksin (18 yaş ve üstü);
abonelik almamış hesaba her gün bir miktar ücretsiz kredi veriliyor.

## Isınma — Bir klip kaç kare?

Video art arda gelen karelerden oluşur. Sinema filmi saniyede 24 kare gösterir; göz kareleri ayrı ayrı değil, hareket olarak görür.

**Çalıştırmadan önce tahmin et, sohbete yaz:** 8 saniyelik bir klip, saniyede 24 kareyle kaç kare eder?

In [ ]:
kare_hizi = 24
sure = 8
kare = sure * kare_hizi
print(sure, "saniye:", kare, "kare")

192 kare. Video modeli bu kareleri tek tek değil, birbirini **tutacak** biçimde birlikte kurar: fincan kare kare şekil
değiştirmemeli, ışık bir anda zıplamamalı. Konu 8'de bir görsel birkaç saniyede geliyordu; 192 tutarlı kare dakikalar sürebilir.
Videonun görselden hem yavaş hem pahalı olmasının kökü burada.

## Adım 1 — Arayüzde ilk klip: başlat ve bırak

`ders_notu.md`'deki **Adım 1** adımlarıyla Google Flow'u aç ve **metinden** bir klip başlat. Yönerge İngilizce
(Konu 7: modeller çoğunlukla İngilizce metinle eğitildi):

> steam rising from a ceramic coffee cup on a wooden desk, soft morning light, slow camera push in

Ayar: **720p, 8 saniye.** Üret düğmesine bastıktan sonra klip hemen gelmez. Sekmeyi açık bırak ve **Adım 2'ye geç**: arayüz
işi arka planda sürdürüyor. Buna **uzun süren iş** denir: isteği başlatırsın, başka bir işe geçersin, sonuç hazır olunca dönersin.

**Sohbete yaz:** düğmeye bastığında ekranda ne gördün: bir yükleme çubuğu, bir yüzde, bir mesaj, hiçbir şey?

Yönergeyi bir yere kopyala; Adım 6'da günlüğe yazacağız.

## Adım 2 — Bir saniye kaç dolar?

Arayüzde para ödemedin; klip ücretsiz kredinden düştü. Aynı klibi **koddan**, ücretli bir video API'siyle üretseydin
her saniyesi için para öderdin. Google'ın resmi fiyat sayfasına göre (28 Ekim 2026) saniye fiyatı **çözünürlüğe** göre değişiyor.

Fiyatları bir sözlüğe koyuyoruz: anahtar çözünürlük, değer bir saniyenin yaklaşık fiyatı (dolar). Python ondalık sayıyı
virgülle değil **noktayla** yazar: `0.101`.

In [ ]:
fiyatlar = {
    "360p": 0.034,
    "720p": 0.101,
    "1080p": 0.152,
    "4K": 0.304,
}

sure = 8
fiyat = fiyatlar["720p"]
ucret = sure * fiyat
print(sure, "saniye, 720p:", ucret, "USD")

Çözünürlük görüntünün piksel boyutu: 720p 1280 × 720, 1080p 1920 × 1080, 4K 3840 × 2160 piksel.
4K, 1080p'nin **dört katı** piksel: eni de boyu da iki katı.

**Çalıştırmadan önce tahmin et, sohbete yaz:** 4K'nın saniye fiyatı 1080p'ninkinin kaç katı? 4 mü, daha az mı, daha çok mu?

In [ ]:
kat = fiyatlar["4K"] / fiyatlar["1080p"]
print("4K, 1080p'nin", kat, "katı fiyat")

İki katı. Fiyat piksel sayısıyla aynı hızda artmıyor; ama her basamak yine de fiyatı belirgin biçimde artırıyor.
Asıl soru "en yüksek kalite hangisi?" değil, "bu klip **nerede** izlenecek?" Çoğu telefon ekranı 4K değil; 4K'nın fazladan
pikselleri orada görünmez, faturada görünür.

## Adım 3 — 20 saniyelik tanıtım: hangi ayar yarısına indirir?

Bir kafe 20 saniyelik bir tanıtım videosu istiyor ve "4K olsun" diyor. Model tek seferde birkaç saniyelik klip üretir;
20 saniye, birkaç klibin kurguda birleşmesidir. Hesap için toplam süre yeter.

In [ ]:
sure = 20
cozunurluk = "4K"
fiyat = fiyatlar[cozunurluk]
ucret = sure * fiyat
print(sure, "saniye,", cozunurluk, ":", ucret, "USD")

Kafenin bütçesi bunun **yarısı**: 3,04 dolar. Önce süreye dokunmadan her çözünürlükte 20 saniyenin bedeline bakalım.
Bir sözlüğün üstünde `for` döngüsü anahtarları sırayla verir (Konu 4'teki `for kelime in harita:` gibi). `round(..., 2)`
virgülden sonra iki basamak bırakır (Konu 3).

In [ ]:
for cozunurluk in fiyatlar:
    fiyat = fiyatlar[cozunurluk]
    ucret = round(sure * fiyat, 2)
    print(cozunurluk, ":", ucret, "USD")

4K'dan 1080p'ye inince süre aynı kalıyor, fiyat yarıya iniyor. Şimdi tersini deneyelim: çözünürlük 4K kalsın, süre kısalsın.
Hangi süreler bütçeye sığar?

In [ ]:
butce = 3.04
sureler = [5, 10, 15, 20]
fiyat = fiyatlar["4K"]
for sure in sureler:
    ucret = round(sure * fiyat, 2)
    if ucret <= butce:
        print(sure, "saniye, 4K:", ucret, "USD, bütçeye sığıyor")
    else:
        print(sure, "saniye, 4K:", ucret, "USD, bütçeyi aşıyor")

Yarı bütçeyle iki yol var: **20 saniye 1080p** ya da **10 saniye 4K.** Hangisini önerirdin? Tanıtım nerede gösterilecek:
telefonda, kafenin ekranında, sinema perdesinde? 10 saniyeye inersen hangi çekimleri atardın? **Sohbete yaz.**

Bu hesap kaç kez denediğini saymıyor. Videoda ilk deneme çoğu zaman istediğin gibi olmaz; üç deneme yaparsan aynı 20 saniyenin
faturası üç katına çıkar. Bu yüzden önce kısa ve düşük çözünürlükte dene, beğendiğini büyüt.

## Adım 4 — Klibe dön: izle ve değerlendir

Arayüze dön: klip hazır mı? Hazırsa izle ve indir (indirme ve dosyayı nereye koyacağın ders notunda: `videolar/metinden.mp4`).
Tasarımcı gözüyle izle ve **sohbete yaz:**

- Yönergede "slow camera push in" (kamera yavaşça yaklaşır) yazdık. Kamera bunu yaptı mı?
- Buhar nasıl hareket ediyor? Nerede inandırıcı, nerede "yapay" duruyor?
- Klipte ses var mı? Yönergede ses yoktu; ne duyuyorsun, sen ne isterdin?
- Üret düğmesine bastıktan kaç saniye sonra geldi? Kredinden ne kadar düştü?

## Adım 5 — Görselden klip

Şimdi Konu 8'de ürettiğin görsellerden birini videonun **ilk karesi** yapıyoruz (arayüz adımları ders notunda, **Adım 5**).
Konu 8'in görselleri `08-koddan-toplu-uretim` klasöründe: `izgara/`, `ciktilar/` ya da ödevinin `odev8/` klasörü.
Görselin yoksa bu klasördeki `veri/yedek_fincan.jpg`'yi kullan.

Yönerge artık sahneyi değil, **ne olacağını** anlatıyor; sahne görselde zaten var. Örnek:

> slow camera push in, gentle natural motion, soft morning light

Ayar: **720p, 6 saniye.** Görselin kare (1:1), video yatay (16:9).

**Üretmeden önce tahmin et, sohbete yaz:** model kenarları ne yapacak? Görseli kırpacak mı, kenarlara yeni bir şey mi uyduracak?

Üret düğmesine bas ve beklerken Adım 6'ya geç: ilk klibin kaydını yaz. Klip gelince karşılaştır:

- Kompozisyon korundu mu? Stil (suluboya, linol, düz vektör...) hareket boyunca korundu mu, yoksa video "fotoğrafa" mı kaydı?
- Hangi görseller harekete uygun: derinliği, boşluğu, hareket edebilecek bir öğesi (buhar, yaprak, su) olanlar mı, düz bir afiş mi?

## Adım 6 — Video günlüğü

Konu 8'de her görselin nasıl üretildiğini bir JSON günlüğüne yazmıştık. Arayüzde de aynı alışkanlık: aylar sonra biri
"şunun bir benzerini yapalım" derse yönerge ve ayarlar elinde olmalı. Arayüzün geçmişi senin dosyan değil; kredi biter,
araç değişir, hesap kapanır.

`input` (Konu 8'de de kullandın) ekranın üstünde bir kutu açar; yazdığını **metin** olarak verir, Enter'a basınca hücre devam
eder. Yönergeyi arayüzden
kopyalayıp kutuya yapıştır. Süreyi, çözünürlüğü ve oranı arayüzde seçtiğinle aynı yap: süre **sayı** (tırnaksız), çözünürlük
fiyat sözlüğündeki gibi (`"720p"`), oran arayüzdeki gibi (`"16:9"`). Başka bir arayüz kullandıysan `"arac"` değerini değiştir.

In [ ]:
yonerge = input("Metinden klibin yönergesi: ")
begendim = input("Beğendin mi? (evet / kısmen / hayır): ")

metin_kaydi = {
    "tur": "metinden",
    "arac": "Google Flow",
    "yonerge": yonerge,
    "gorsel": "yok",
    "sure": 8,
    "cozunurluk": "720p",
    "oran": "16:9",
    "dosya": "videolar/metinden.mp4",
    "begendim": begendim,
}
print(metin_kaydi)

Görselden klibin kaydı aynı kalıpta. `"gorsel"` alanına kullandığın görselin yolunu yaz. Konu 8'deki bir görselse yol
klasörün adıyla başlasın, ör. `"08-koddan-toplu-uretim/izgara/suluboya_gece.jpg"`.

In [ ]:
yonerge = input("Görselden klibin yönergesi: ")
begendim = input("Beğendin mi? (evet / kısmen / hayır): ")

gorsel_kaydi = {
    "tur": "gorselden",
    "arac": "Google Flow",
    "yonerge": yonerge,
    "gorsel": "veri/yedek_fincan.jpg",
    "sure": 6,
    "cozunurluk": "720p",
    "oran": "16:9",
    "dosya": "videolar/gorselden.mp4",
    "begendim": begendim,
}
print(gorsel_kaydi)

İki kaydı bir listeye koyup dosyaya yazıyoruz; Konu 8'deki günlüğün aynısı. `ensure_ascii=False` Türkçe harfleri olduğu gibi
yazar, `indent=2` dosyayı okunur kılar.

In [ ]:
import json

kayitlar = [metin_kaydi, gorsel_kaydi]
with open("video_gunlugu.json", "w", encoding="utf-8") as dosya:
    json.dump(kayitlar, dosya, ensure_ascii=False, indent=2)

sayi = len(kayitlar)
print(sayi, "kayıt video_gunlugu.json dosyasına yazıldı")

Son olarak günlüğü geri okuyup soruyoruz: bu klipler ücretli bir API'yle koddan üretilseydi kaça mal olurdu? Her kaydın
çözünürlüğü fiyat sözlüğünde bir anahtar.

In [ ]:
with open("video_gunlugu.json", encoding="utf-8") as dosya:
    okunan = json.load(dosya)

toplam = 0
for kayit in okunan:
    cozunurluk = kayit["cozunurluk"]
    fiyat = fiyatlar[cozunurluk]
    ucret = kayit["sure"] * fiyat
    toplam = toplam + ucret

toplam = round(toplam, 2)
print("Bu klipler ücretli bir API'yle:", toplam, "USD")

Kendi süre ve çözünürlüğünü yazdıysan senin sayın farklı olabilir. Az bir para gibi görünüyor; ama bu yalnızca **iki klip**,
her biri tek denemede, tek bir kişi için. 12 kişilik bir sınıfta herkes birkaç deneme yapsa fatura hızla büyür. Bugün sıfır
bütçeyle video yapabiliyoruz, çünkü arayüzün ücretsiz günlük kredisi var. Kredi bitince ya ertesi günü beklersin ya da ödersin.

## Bonus

**1. Ücretsiz kredini planla.** Google Flow ücretsiz hesaba her gün 50 kredi veriyor; bir klibin kaç kredi tuttuğu süreye ve
çözünürlüğe bağlı. 28 Ekim 2026'da 720p bir klip için: 4 saniye 7, 6 saniye 10, 8 saniye 12, 10 saniye 15 kredi. Arayüzde
gördüğün sayılar farklıysa sözlüğü düzelt.

In [ ]:
krediler = {4: 7, 6: 10, 8: 12, 10: 15}
for sure in krediler:
    kredi = krediler[sure]
    saniye_basi = round(kredi / sure, 2)
    print(sure, "saniye:", kredi, "kredi, saniyesi", saniye_basi, "kredi")

Uzun klibin saniyesi daha ucuz; ama denemede kısa klip toplamda daha az harcar. 50 krediyle kaç tane 4 saniyelik, kaç tane
10 saniyelik klip üretebilirsin? Bir günün kredisini nasıl bölerdin: kaç deneme, kaç son hâl?

**2. Dikey video.** Metinden klibin yönergesini arayüzde 9:16 oranında (telefon, hikâye) bir kez daha üret. Kompozisyon nasıl
yeniden kuruldu: fincan nerede duruyor, ne kırpıldı, ne eklendi? Günlüğe üçüncü kayıt olarak ekle: Adım 6'nın ilk hücresini
kopyala, sözlüğün adını `dikey_kaydi`, `"oran"`ı `"9:16"`, `"dosya"`yı `"videolar/dikey.mp4"` yap; `kayitlar` satırını
`[metin_kaydi, gorsel_kaydi, dikey_kaydi]` yap ve son iki hücreyi yeniden çalıştır.